# Muhtemel Ask - 03 FINALIZE V2 (BETA)

Run only after `02_ALIGN_PREPARE_ID.ipynb` and after the exact Indonesian output ZIP has been placed in `translation_output/`. The finalizer independently rebuilds the aligned schema and speech coverage from the full raw-ASR and forced-alignment artifacts, reruns timing and semantic QA, and writes the V2 PASS report last. It never overwrites the V1 report or V1 output names.

In [ ]:
EPISODE = 12  # @param {type:"integer"}

if isinstance(EPISODE, bool) or not isinstance(EPISODE, int) or EPISODE < 1:
    raise ValueError("EPISODE must be a positive integer")

## Mount Drive and load only the isolated V2 runtime

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import importlib
import shutil
import subprocess
import sys

SYSTEM_ROOT = Path("/content/drive/MyDrive/Muhtemel_Ask_Subtitles/SYSTEM_V2_BETA")
REQUIREMENTS_PATH = SYSTEM_ROOT / "requirements-v2-colab.txt"
if not (SYSTEM_ROOT / "src/finalize_v2.py").is_file() or not REQUIREMENTS_PATH.is_file():
    raise FileNotFoundError("The complete isolated V2 SYSTEM folder is missing from Drive")
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS_PATH)],
    check=True,
)
if str(SYSTEM_ROOT) not in sys.path:
    sys.path.insert(0, str(SYSTEM_ROOT))
for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
importlib.invalidate_caches()
print("Pinned V2 finalization runtime ready.")

## Resolve the exact episode evidence and translated output

In [ ]:
import json
import yaml

from src.download import load_valid_stage_marker

SERIES_CONFIG_PATH = SYSTEM_ROOT / "config/series.yaml"
NAMES_CONFIG_PATH = SYSTEM_ROOT / "config/names.yaml"
RELIGIOUS_CONFIG_PATH = SYSTEM_ROOT / "config/religious_terms.yaml"
for path, label in (
    (SERIES_CONFIG_PATH, "series config"),
    (NAMES_CONFIG_PATH, "names config"),
    (RELIGIOUS_CONFIG_PATH, "religious config"),
):
    if path.is_symlink() or not path.is_file() or path.stat().st_size <= 0:
        raise FileNotFoundError(f"Canonical {label} is missing or unsafe: {path}")
with SERIES_CONFIG_PATH.open(encoding="utf-8") as handle:
    series_config = yaml.safe_load(handle)
if not isinstance(series_config, dict):
    raise RuntimeError("series.yaml root must be a mapping")

EPISODE_NAME = f"Muhtemel Ask {EPISODE}.Bolum"
DRIVE_ROOT = Path(series_config["drive_root"])
EPISODE_ROOT = DRIVE_ROOT / "EPISODES" / EPISODE_NAME
DIRS = {name: EPISODE_ROOT / name for name in (
    "source", "prepare", "translation_input", "translation_output"
)}
if EPISODE_ROOT.is_symlink() or not EPISODE_ROOT.is_dir():
    raise FileNotFoundError(f"Exact episode folder is missing or unsafe: {EPISODE_ROOT}")
for name, folder in DIRS.items():
    if folder.is_symlink() or not folder.is_dir():
        raise FileNotFoundError(f"Required episode directory is missing or unsafe: {name}")

DOWNLOAD_MARKER_PATH = DIRS["source"] / "download.done.json"
with DOWNLOAD_MARKER_PATH.open(encoding="utf-8") as handle:
    raw_download_marker = json.load(handle)
download_marker = load_valid_stage_marker(
    DOWNLOAD_MARKER_PATH,
    stage="download",
    input_sha256=raw_download_marker.get("input_sha256", ""),
    required_output_keys=("video", "metadata"),
    optional_output_keys=("captions",),
    allowed_root=DIRS["source"],
)
if download_marker is None:
    raise RuntimeError("Download source marker or source hashes are invalid")
SOURCE_VIDEO = Path(download_marker["outputs"]["video"]["path"]).resolve()
if SOURCE_VIDEO.parent != DIRS["source"].resolve() or SOURCE_VIDEO.stem != EPISODE_NAME:
    raise RuntimeError("Download marker does not identify the exact episode-named source")

RAW_ASR_PATH = DIRS["prepare"] / "raw_asr_v2.json"
FORCED_ALIGNMENT_PATH = DIRS["prepare"] / "forced_alignment_v2.json"
SCHEMA_PATH = DIRS["prepare"] / "aligned_tr_schema_v2.json"
TR_PACK_PATH = DIRS["translation_input"] / f"{EPISODE_NAME}_TR_CORRECTION_PACK.zip"
TR_TEXT_OUTPUT_PATH = DIRS["translation_output"] / f"{EPISODE_NAME}_TR_TEXT_CORRECTED.zip"
TR_OUTPUT_PATH = DIRS["translation_output"] / f"{EPISODE_NAME}_TR_CORRECTED.zip"
AUDIO_REVIEW_PATH = DIRS["prepare"] / "audio_review_v2.json"
ID_PACK_PATH = DIRS["translation_input"] / f"{EPISODE_NAME}_ID_TRANSLATION_PACK.zip"
ID_OUTPUT_PATH = DIRS["translation_output"] / f"{EPISODE_NAME}_ID_TRANSLATED.zip"
for path, label in (
    (RAW_ASR_PATH, "raw ASR V2"),
    (FORCED_ALIGNMENT_PATH, "full forced alignment"),
    (SCHEMA_PATH, "aligned Turkish schema"),
    (TR_PACK_PATH, "Turkish correction pack"),
    (TR_TEXT_OUTPUT_PATH, "text-only Turkish correction output"),
    (TR_OUTPUT_PATH, "audio-reviewed Turkish correction output"),
    (AUDIO_REVIEW_PATH, "bounded Colab audio-review report"),
    (ID_PACK_PATH, "Indonesian translation pack"),
    (ID_OUTPUT_PATH, "Indonesian translated output"),
):
    if path.is_symlink() or not path.is_file() or path.stat().st_size <= 0:
        raise FileNotFoundError(f"Exact {label} is missing or unsafe: {path}")
print(f"All canonical V2 evidence exists for {EPISODE_NAME}.")

## Rebuild, verify, stream-copy mux, and commit V2 PASS

In [ ]:
import src.finalize_v2 as finalize_v2_module

module_path = Path(finalize_v2_module.__file__).resolve()
if SYSTEM_ROOT.resolve() not in module_path.parents:
    raise RuntimeError(f"Loaded finalize_v2 from an unexpected runtime: {module_path}")
report = finalize_v2_module.finalize_episode_v2(
    episode_root=EPISODE_ROOT,
    episode=EPISODE,
    source_video=SOURCE_VIDEO,
    raw_asr_path=RAW_ASR_PATH,
    forced_alignment_path=FORCED_ALIGNMENT_PATH,
    tr_correction_pack=TR_PACK_PATH,
    tr_text_correction_output=TR_TEXT_OUTPUT_PATH,
    tr_correction_output=TR_OUTPUT_PATH,
    audio_review_path=AUDIO_REVIEW_PATH,
    aligned_schema=SCHEMA_PATH,
    id_translation_pack=ID_PACK_PATH,
    id_translation_zip=ID_OUTPUT_PATH,
    series_config=SERIES_CONFIG_PATH,
    names_config=NAMES_CONFIG_PATH,
    religious_config=RELIGIOUS_CONFIG_PATH,
)

## Verify the canonical V2 publication

In [ ]:
REPORT_PATH = EPISODE_ROOT / "final" / f"{EPISODE_NAME}_FINALIZATION_REPORT_V2.json"
with REPORT_PATH.open(encoding="utf-8") as handle:
    persisted_report = json.load(handle)
if persisted_report != report or report.get("status") != "PASS" or report.get("report_version") != 2:
    raise RuntimeError("V2 finalization report read-back or PASS identity failed")
expected_outputs = {
    "mkv": f"final/{EPISODE_NAME}.mkv",
    "id_srt": f"final/subtitles/{EPISODE_NAME}-id.srt",
    "tr_srt": f"final/subtitles/{EPISODE_NAME}-tr.srt",
}
for key, relative_path in expected_outputs.items():
    if report["outputs"][key]["relative_path"] != relative_path:
        raise RuntimeError(f"Unexpected V2 {key} output path")
    if not (EPISODE_ROOT / relative_path).is_file():
        raise FileNotFoundError(f"Published V2 {key} is missing")
alignment_review_count = report["alignment_policy_v2"]["review_alignment_score_count"]
audio_review = report["audio_review_v2"]
print("FINALIZATION V2 PASS")
print(f"Report: {REPORT_PATH}")
print(f"PILOT WARNING ONLY (PASS failure değildir): review_alignment_score_count={alignment_review_count}")
print("Exact audio-review outcomes:")
print(f"  confirmed_dialogue: {audio_review['confirmed_dialogue_count']}")
print(f"  reviewed_non_dialogue: {audio_review['reviewed_non_dialogue_count']}")
print(f"  discarded_asr_hallucination: {audio_review['discarded_asr_hallucination_count']}")
print(f"  pending_audio_review: {audio_review['pending_audio_review_count']}")
for key in ("mkv", "id_srt", "tr_srt"):
    print(f"  {key}: {report['outputs'][key]['relative_path']}")